# Document Analysis with predict-rlm

This notebook walks through building an RLM that analyzes PDF documents and produces structured reports. We'll go step by step — defining the output schema, the task signature, the PDF skill, and then running the RLM on a real document.

The key idea: we pass `File` references to the RLM, which mounts the PDFs into its sandbox. The RLM opens them directly with pymupdf, renders pages as images, and uses `predict()` with vision to extract structured information — autonomously deciding what to look at and when.

## Step 1: Define the output schema

We start by defining what we want the RLM to produce. Pydantic models give us typed, validated outputs — not free-form text. The `Field(description=...)` annotations tell the RLM what each field means.

In [9]:
from pydantic import BaseModel, Field


class KeyDate(BaseModel):
    """A key date extracted from a document."""
    name: str = Field(description="e.g. 'Submission Deadline', 'Effective Date'")
    date: str = Field(description="ISO format date (YYYY-MM-DD)")
    time: str | None = Field(None, description="24-hour format (HH:MM)")
    timezone: str | None = Field(None, description="Timezone code, e.g. 'EST', 'UTC'")


class KeyEntity(BaseModel):
    """A key entity extracted from a document."""
    name: str = Field(description="Name of the person, organization, or role")
    role: str | None = Field(None, description="Role or relationship to the document")
    contact: str | None = Field(None, description="Contact info if available")


class DocumentAnalysis(BaseModel):
    """Structured analysis of a document set."""
    report: str = Field(description="Full analysis as a well-formatted markdown report")
    key_dates: list[KeyDate] = Field(default_factory=list, description="Important dates found in the documents")
    key_entities: list[KeyEntity] = Field(
        default_factory=list, description="Key people, organizations, or roles mentioned"
    )

## Step 2: Define the task signature

The DSPy signature tells the RLM three things:
1. **What it receives** — `list[File]` references to the PDFs, which are mounted into the sandbox automatically
2. **What it should produce** — our `DocumentAnalysis` schema
3. **How to approach the task** — the docstring becomes the RLM's step-by-step instructions

In [10]:
import dspy

from predict_rlm import File, Skill


class AnalyzeDocuments(dspy.Signature):
    """Analyze documents and produce a structured report.

    1. **Read the report criteria** (appended below) to understand what
       information to extract and in what format. The criteria define the
       structure and sections of the report.

    2. **Survey the documents** to understand what you're working with:
       file names, page counts, document types (main document,
       attachments, amendments, etc.).

    3. **Gather information** systematically by rendering pages as images
       and using predict() to extract relevant content. Work through the
       criteria's requested sections, locating the relevant information
       in the documents.

    4. **Produce the report** following the format specified in the criteria.
       Use tables for structured data, prose for analysis and context.
    """

    documents: list[File] = dspy.InputField(
        desc="PDF documents to analyze"
    )
    analysis: DocumentAnalysis = dspy.OutputField(
        desc="Structured analysis with markdown report, key dates, and key entities"
    )

## Step 3: Define the PDF skill

The sandbox starts with just Python's standard library and `predict()`. Skills add packages and domain knowledge on top. Here we give the RLM `pymupdf` for PDF processing, plus instructions on how to use it — preferring visual rendering over raw text extraction for analysis.

In [11]:
pdf_skill = Skill(
    name="pdf",
    instructions="""Use pymupdf to work with PDF files mounted in the sandbox.

## Opening and inspecting

    import pymupdf
    doc = pymupdf.open(path)
    print(f"Pages: {len(doc)}")
    toc = doc.get_toc()  # [[level, title, page_num], ...]
    metadata = doc.metadata  # dict with title, author, subject, etc.

## Reading pages — prefer visual rendering over raw text

Always render pages as images for analysis with predict(). Raw text
extraction loses layout, tables, headers, and formatting that are
critical for understanding documents. Use get_text() only for
keyword searches or when you need to find specific strings.

Render a page as an image:

    import base64
    pix = doc[page_num].get_pixmap(dpi=200)
    uri = f"data:image/png;base64,{base64.b64encode(pix.tobytes('png')).decode()}"
    result = await predict("page: dspy.Image -> ...", page=uri)

Render multiple pages in parallel:

    import asyncio, base64

    def render_page(doc, i):
        pix = doc[i].get_pixmap(dpi=200)
        return f"data:image/png;base64,{base64.b64encode(pix.tobytes('png')).decode()}"

    images = [render_page(doc, i) for i in range(len(doc))]
    tasks = [predict("page: dspy.Image -> ...", page=img) for img in images]
    results = await asyncio.gather(*tasks)

## Text extraction (for searching, not analysis)

    text = doc[page_num].get_text()           # plain text
    blocks = doc[page_num].get_text_blocks()   # [(x0,y0,x1,y1,text,block_no,type), ...]
    words = doc[page_num].get_text_words()     # [(x0,y0,x1,y1,word,block,line,word_no), ...]

## Searching

    results = doc[page_num].search_for("keyword")  # list of Rect objects

## Table extraction

    tables = doc[page_num].find_tables()
    for table in tables:
        data = table.extract()  # list of lists (rows x cols)

## Images

    images = doc[page_num].get_images()       # [(xref, smask, w, h, bpc, cs, ...), ...]
    img_bytes = doc.extract_image(xref)        # dict with "image" (bytes), "ext", etc.

## Annotations and links

    links = doc[page_num].get_links()          # [{"kind": ..., "uri": ..., ...}, ...]
    annots = list(doc[page_num].annots())      # annotation objects

## Writing and modifying

    doc[page_num].insert_text((x, y), "text")
    doc[page_num].add_redact_annot(rect)
    doc[page_num].apply_redactions()
    doc.save("/sandbox/output/field_name/modified.pdf")
    doc.close()

Always close the document when done.""",
    packages=["pymupdf"],
)

## Step 4: Load documents and build the RLM

In [12]:
from pathlib import Path

import pymupdf

from predict_rlm import PredictRLM

# Load the sample document
pdf_path = Path("sample/input/YYJ-2025-Parking-Management-RFP.pdf").resolve()
documents = [File(path=str(pdf_path))]

with pymupdf.open(str(pdf_path)) as doc:
    total_pages = len(doc)

print(f"Loaded: {pdf_path.name} ({total_pages} pages, {pdf_path.stat().st_size / 1024 / 1024:.1f} MB)")

# What we want the RLM to extract
CRITERIA = """
Analyze the document(s) and produce a comprehensive briefing report
structured as follows.

---

**Formatting guidelines:**

The report should be professional, easy to read, and visually elegant.
Mix prose, tables, bullets, and numbered items to draw the topology of
the information and help the reader quickly parse the report. Do not
include page references — present information directly. Use bold
sparingly. Favor prose over bullets; use bullets very sparingly.

---

**Report sections:**

1.  **Executive Summary**
    What is this document about? Who are the key parties involved?
    What are the most important facts, decisions, or actions described?

2.  **Key Dates and Timeline**
    All significant dates mentioned in the document: deadlines,
    effective dates, milestones, meetings, expiration dates. Present
    in chronological order. Flag any unusually tight timelines.

3.  **Key Entities and Stakeholders**
    People, organizations, and roles mentioned in the document.
    For each, note their role and relevance. Include contact
    information where available.

4.  **Financial Information**
    Any monetary amounts, fees, budgets, pricing structures,
    payment terms, or financial obligations mentioned. Summarize
    in a table if multiple items exist.
""".strip()

# Append criteria to signature instructions
signature = AnalyzeDocuments.with_instructions(
    AnalyzeDocuments.instructions + "\n\n# Task\n\n" + CRITERIA
)

# Build the RLM
rlm = PredictRLM(
    signature,
    lm="openai/gpt-5.4",
    sub_lm="openai/gpt-5.1",
    skills=[pdf_skill],
    max_iterations=30,
    verbose=True,
)

Loaded: YYJ-2025-Parking-Management-RFP.pdf (136 pages, 5.1 MB)


## Step 5: Run the RLM

In [13]:
import time

start = time.perf_counter()
result = await rlm.acall(documents=documents)
duration = time.perf_counter() - start

analysis = result.analysis
mins, secs = divmod(int(duration), 60)
print(f"Done in {mins}m {secs}s")

2026/04/01 11:24:26 INFO dspy.predict.rlm: RLM iteration 1/30
Reasoning: First iteration should explore the input and document structure before extracting. I need to inspect the PDF: filename, page count, metadata, TOC, and maybe a small text sample from the first few pages to understand document type and likely relevant sections. Then I can render/select pages strategically for model extraction.
Code:
```repl
import os, json, re
import pymupdf

print("documents:", documents)

pdf_path = documents[0]
doc = pymupdf.open(pdf_path)

print("File:", os.path.basename(pdf_path))
print("Pages:", len(doc))
print("Metadata:", json.dumps(doc.metadata, indent=2))
toc = doc.get_toc()
print("TOC entries:", len(toc))
print("TOC sample:", toc[:10])

# Print short text previews from first few pages for orientation
for i in range(min(5, len(doc))):
    text = doc[i].get_text("text")
    text = re.sub(r"\s+", " ", text).strip()
    print(f"\n--- Page {i+1} preview ---")
    print(text[:800])

doc.close()

Done in 2m 12s


## Results

### Report

In [14]:
from IPython.display import Markdown

Markdown(analysis.report)

# Briefing Report — YYJ 2025 Parking Management Services RFP

## Executive Summary

This document set is a Request for Proposals issued by the Victoria Airport Authority for parking management services at Victoria International Airport (YYJ). It solicits proposals from qualified parking operators to manage airport parking and related curb-management functions, with service commencement targeted for June 1, 2025. The package includes the main RFP, schedules, appendices, operational attachments, and a draft contract.

The key parties are the Victoria Airport Authority as issuer and prospective owner-counterparty, David Parson as the Authority's representative for the procurement, and prospective proponents with at least five years in the parking management business. The selected proponent would become the contractor responsible for operating the parking facilities under the draft agreement attached to the RFP.

The most important actions are procedural and time-sensitive. Proponents must submit an intent/receipt confirmation and RSVP by February 20, 2025, attend a mandatory pre-bid meeting and site tour on February 27, 2025, submit written questions by March 6, 2025, and deliver proposals by March 31, 2025 at 2:00 pm PST. The Authority indicates an anticipated award on April 24, 2025 and contract commencement on June 1, 2025.

Commercially, the RFP is not framed around a fixed disclosed budget. Instead, the draft contract centers on a management-fee model, with GST addressed separately, contractor invoicing and monthly accounting obligations, and security requirements including an irrevocable letter of credit equal to 5% of the total management fee for the initial term. The draft contract also references interest on overdue amounts at 2% per month compounded monthly (26.824% per year). Evaluation in the RFP gives substantial weight to pricing, but also weighs understanding and approach, compliance with operational requirements, First Nations opportunities, and value-add.

## Key Dates and Timeline

| Date | Time | Item | Notes |
|---|---:|---|---|
| 2025-02-13 | — | Date of Issue | RFP released by Victoria Airport Authority. |
| 2025-02-20 | 14:00 PST | Receipt Confirmation / Intent to Submit / RSVP due | Early administrative deadline. This is relatively tight from issuance. |
| 2025-02-20 | 14:00 PST | RSVP for mandatory pre-bid meeting and site tour | Same-day deadline as receipt confirmation. |
| 2025-02-27 | 13:30 PST | Mandatory pre-bid meeting and site tour | Attendance is mandatory; non-attendees are excluded from bidding. |
| 2025-03-06 | 14:00 PST | Deadline for Questions | Final date for written inquiries to the Authority's representative. |
| 2025-03-10 | — | Question Response Deadline | Authority's target date for responses. |
| 2025-03-31 | 14:00 PST | Closing Date for Submission of Proposals | Formal proposal deadline. |
| 2025-04-24 | — | Award to Successful Proponent (if any) | Stated as anticipated, not guaranteed. |
| 2025-06-01 | — | Contract Commencement | Target start date for services. |

The schedule is fairly compressed at the front end. Only one week separates the RSVP deadline from the mandatory site visit, and less than five weeks separate issuance from proposal close. Any bidder that misses the February 27 mandatory meeting is effectively disqualified.

The RFP also states that proposals remain valid for acceptance for sixty days following the closing date. In addition, unsuccessful proponents may request a debriefing within thirty days of notification.

## Key Entities and Stakeholders

| Entity | Role / Relevance | Contact |
|---|---|---|
| Victoria Airport Authority | Issuer of the RFP and owner-side contracting authority. | 201-1640 Electra Blvd, Sidney, BC V8L 5V4 |
| Victoria International Airport (YYJ) | Site where parking management and curb-management services will be performed. | Victoria International Airport, Sidney, BC |
| David Parson | Commercial Development Officer and named Authority representative for this procurement. All inquiries are to be directed to him or his designate. | (250) 413-7897; rfpparkingmgmt@yyj.ca |
| Proponent | Any eligible bidder responding to the RFP. Must satisfy pre-qualification requirements. | Not specified |
| Successful Proponent / Contractor | The bidder awarded the contract and responsible for performance under the resulting agreement. | Not specified |
| Authority's Authorized Representative | VAA-designated representative for RFP and contract administration. | Not separately identified in the extracted pages beyond the issuing-office contact |

The RFP places strong procedural control on communications. Proponents are directed to communicate in writing only through the named issuing-office contact, and the Authority disclaims reliance on statements made by other persons. Operationally, the contractor will be a visible service provider at the airport and is expected to manage both parking operations and related customer-facing functions.

## Financial Information

The document set does not disclose a single project budget. Instead, it establishes a contractual commercial framework under which the successful proponent will be paid a management fee and must comply with security, invoicing, and accounting obligations.

| Financial Item | Details |
|---|---|
| Pricing evaluation weight | 30% of the evaluation score. |
| Management fee | The draft contract includes a management-fee structure in Part 2.1. Exact dollar values are to be proposed/inserted rather than pre-disclosed in the RFP text reviewed. |
| GST | Addressed separately in the contract under Part 2.2. |
| Letter of credit / performance security | Successful proponent must provide an irrevocable, unconditional letter of credit equal to 5% of the total management fee for the initial term of the contract. |
| Interest on overdue amounts | 2% per month, compounded monthly, stated as 26.824% per year in the contract definitions. |
| Invoicing / monthly accounting | The draft contract references monthly statements of account and invoice-related obligations, indicating recurring reporting and payment administration requirements. |
| Proposal / participation costs | Onsite presentations, if requested after the proposal due date, are at the proponent's cost. |
| Insurance | The successful proponent must demonstrate insurability and comply with the insurance requirements in the draft contract's contract security and insurance schedule. Specific extracted pages reviewed referenced the requirement, though not all coverage limits were captured in the current extraction set. |

From a bidder's perspective, the most material financial features are the management-fee pricing proposal, the 5% letter-of-credit requirement tied to the initial term's total management fee, and the contract's strict payment/interest framework. From the Authority's perspective, the structure appears designed to preserve financial control through monthly reporting, formal invoicing, security, and insurance-backed performance.


### Structured data

In [15]:
print("KEY DATES")
print("-" * 60)
for d in analysis.key_dates:
    time_str = f" at {d.time}" if d.time else ""
    tz_str = f" {d.timezone}" if d.timezone else ""
    print(f"  {d.date}{time_str}{tz_str} — {d.name}")

print()
print("KEY ENTITIES")
print("-" * 60)
for e in analysis.key_entities:
    role = f" ({e.role})" if e.role else ""
    contact = f" — {e.contact}" if e.contact else ""
    print(f"  {e.name}{role}{contact}")

KEY DATES
------------------------------------------------------------
  2025-02-13 — Date of Issue
  2025-02-20 at 14:00 PST — Receipt Confirmation / Intent to Submit / RSVP Form Due
  2025-02-20 at 14:00 PST — RSVP for Pre-Bid Meeting and Site Tour
  2025-02-27 at 13:30 PST — Mandatory Pre-Bid Meeting and Site Tour
  2025-03-06 at 14:00 PST — Deadline for Questions
  2025-03-10 — Question Response Deadline
  2025-03-31 at 14:00 PST — Closing Date for Submission of Proposals
  2025-04-24 — Award to Successful Proponent (if any)
  2025-06-01 — Contract Commencement

KEY ENTITIES
------------------------------------------------------------
  Victoria Airport Authority (Issuer of the RFP and contracting authority for parking management services at Victoria International Airport) — 201-1640 Electra Blvd, Sidney, BC V8L 5V4
  Victoria International Airport (YYJ) (Airport where the services will be performed) — 1640 Electra Blvd, Sidney, BC
  David Parson (Commercial Development Officer; Au

### Run stats

In [16]:
lm_history = list(rlm._lm.history)
sub_lm_history = list(rlm._sub_lm.history)

lm_cost = sum(e.get("cost", 0) or 0 for e in lm_history)
lm_input = sum(e.get("usage", {}).get("prompt_tokens", 0) or 0 for e in lm_history)
lm_output = sum(e.get("usage", {}).get("completion_tokens", 0) or 0 for e in lm_history)

sub_lm_cost = sum(e.get("cost", 0) or 0 for e in sub_lm_history)
sub_lm_input = sum(e.get("usage", {}).get("prompt_tokens", 0) or 0 for e in sub_lm_history)
sub_lm_output = sum(e.get("usage", {}).get("completion_tokens", 0) or 0 for e in sub_lm_history)

total_cost = lm_cost + sub_lm_cost

print(f"Main LM ({len(lm_history)} calls):")
print(f"  Input:  {lm_input:,} tokens")
print(f"  Output: {lm_output:,} tokens")
print(f"  Cost:   ${lm_cost:.4f}")
print()
print(f"Sub-LM ({len(sub_lm_history)} calls):")
print(f"  Input:  {sub_lm_input:,} tokens")
print(f"  Output: {sub_lm_output:,} tokens")
print(f"  Cost:   ${sub_lm_cost:.4f}")
print()
print(f"Total:     ${total_cost:.4f} ({total_pages} pages, ${total_cost / max(total_pages, 1):.4f}/page)")
print(f"Duration:  {mins}m {secs}s")

Main LM (5 calls):
  Input:  40,483 tokens
  Output: 6,446 tokens
  Cost:   $0.1403

Sub-LM (57 calls):
  Input:  58,720 tokens
  Output: 11,989 tokens
  Cost:   $0.1933

Total:     $0.3336 (136 pages, $0.0025/page)
Duration:  2m 12s
